# 04 - Modeling

Pipeline:
1. Baseline scores on **raw** data (no preprocessing) - the reference point.
2. Full preprocessing (`src.feature_engineering.data_prep`) applied independently to train/test to avoid leakage.
3. Baseline scores on **preprocessed** data.
4. Hyperparameter tuning (GridSearchCV) for KNN, CART, RF, XGBoost, LightGBM.
5. Soft-voting ensemble (KNN + RF + LightGBM).

In [1]:
import sys
sys.path.append("..")

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
from sklearn.model_selection import train_test_split

from src.feature_engineering import data_prep
from src.modeling import base_models, fit_models

In [2]:
df = pd.read_csv("../data/diabetes.csv")
X = df.drop(["Outcome"], axis=1)
y = df["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

## Baseline - raw data

Fitting models before any preprocessing gives the reference point used later to measure the impact of preprocessing.

In [3]:
base_models(X_train, y_train, scoring="accuracy")

Base Models (accuracy)....
accuracy: 0.7785 (LR)
accuracy: 0.7411 (KNN)
accuracy: 0.6694 (CART)
accuracy: 0.7655 (RF)
accuracy: 0.7525 (GBM)
accuracy: 0.7329 (XGBoost)
[LightGBM] [Info] Number of positive: 171, number of negative: 320
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000223 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 577
[LightGBM] [Info] Number of data points in the train set: 491, number of used features: 8
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.348269 -> initscore=-0.626657
[LightGBM] [Info] Start training from score -0.626657
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with po

## Preprocessing (train/test independently, no leakage)

In [4]:
X_train, y_train = data_prep(X_train, y_train)
X_test, y_test = data_prep(X_test, y_test)

## Baseline - preprocessed data

In [5]:
base_models(X_train, y_train, scoring="accuracy")

Base Models (accuracy)....
accuracy: 0.8746 (LR)
accuracy: 0.8534 (KNN)
accuracy: 0.8242 (CART)
accuracy: 0.8828 (RF)
accuracy: 0.8762 (GBM)
accuracy: 0.8794 (XGBoost)
[LightGBM] [Info] Number of positive: 171, number of negative: 320
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000105 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 596
[LightGBM] [Info] Number of data points in the train set: 491, number of used features: 19
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.348269 -> initscore=-0.626657
[LightGBM] [Info] Start training from score -0.626657
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gai

## Hyperparameter tuning + voting ensemble

In [6]:
voting_clf, best_models = fit_models(X_train, y_train)

Base Models (accuracy)....
accuracy: 0.8746 (LR)
accuracy: 0.8534 (KNN)
accuracy: 0.8273 (CART)
accuracy: 0.8909 (RF)
accuracy: 0.8762 (GBM)
accuracy: 0.8794 (XGBoost)
[LightGBM] [Info] Number of positive: 171, number of negative: 320
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000140 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 596
[LightGBM] [Info] Number of data points in the train set: 491, number of used features: 19
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.348269 -> initscore=-0.626657
[LightGBM] [Info] Start training from score -0.626657
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with p

## Persist the model + test set

Used by `notebooks/05_evaluation.ipynb` and by `evaluate.py` (terminal evaluation report).

In [7]:
import joblib
lgbm_model = best_models['LightGBM'].fit(X_train, y_train)
joblib.dump(lgbm_model, "../models/lgbm_model.pkl")
joblib.dump(voting_clf, "../models/voting_clf.pkl")

X_test.to_csv("../data/X_test.csv", index=False)
y_test.to_csv("../data/y_test.csv", index=False)

[LightGBM] [Info] Number of positive: 214, number of negative: 400
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000292 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 682
[LightGBM] [Info] Number of data points in the train set: 614, number of used features: 21
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.348534 -> initscore=-0.625489
[LightGBM] [Info] Start training from score -0.625489
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best